# RAG end to end: query rewriting

This notebook builds on `Broken-RAG.ipynb`. The only new RAG step is query rewriting: the user's conversational request is turned into a more general information-retrieval query before searching Doug's blog.

In [3]:
from copy import deepcopy
import numpy as np
from IPython.display import HTML, Markdown, display
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.doug_blog_data import corpus
import os

os.environ["CHEAT_AT_SEARCH_DATA_PATH"] = "/home/jovyan/data"

from cheat_at_search.data_dir import mount
mount(use_gdrive=False)    # colab, share data across notebook runs on gdrive

openai = OpenAI(api_key=key_for_provider('openai'))

You're going to be prompted for your API key. This will be stored in a local file
If you'd prefer to set it as an environment variable, set it as:
    export OPENAI_API_KEY=your_api_key_here


Enter your openai_api_key:  ········


## Search index

In [4]:
def no_chunking(doc):
    yield doc

def chunk_by_paragraph(doc):
    yield from doc.split('\n\n')

def top_n(vectors, query_vector, k=3):
    similarities = np.dot(vectors, query_vector)
    return np.argsort(similarities)[-k:][::-1]

class SearchIndex:
    def __init__(self, corpus, chunk_fn=no_chunking):
        self.corpus = corpus
        self.chunk_fn = chunk_fn
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.all_chunks = []
        self.index = None

    def chunks(self):
        for _, row in self.corpus.iterrows():
            for chunk in self.chunk_fn(row['description']):
                yield row['title'], chunk

    def build_index(self):
        self.all_chunks = list(self.chunks())
        self.index = self.model.encode(
            [chunk for _, chunk in self.all_chunks],
            convert_to_numpy=True,
            show_progress_bar=True,
        )

    def search(self, query, k=3):
        query_vector = self.model.encode([query], convert_to_numpy=True)[0]
        return [self.all_chunks[i] for i in top_n(self.index, query_vector, k)]

## Rewrite conversational requests

The rewrite is intentionally narrow. It removes the request to speak as Doug and produces a general information-retrieval question that should be easier to match against blog content.

In [8]:
answer_system_prompt = '''
You pretend to be search engine expert Doug Turnbull. Answer questions
in his voice using the supplied snippets from Doug's blog. Give Doug's
specific perspective, not a generic answer. If the snippets do not
support a claim, say that the blog context does not establish it.
'''

query_system_prompt = '''
The user is asking Doug Turnbull a question. Reformulate the user's
request as a general information-retrieval question to search Doug's blog.
Remove Doug's name and any request to answer in Doug's voice. Return only
the rewritten search query.
'''

guardrail_system_prompt = """
You are a guardrail for a search engine expert named Doug Turnbull. Your job is to determine if the user's question is about search or information retrieval.

Detail your reasoning for why the query is / is not about information retrieval

Finish your response with either "BLOCKED" or "ALLOWED" to indicate if the question is allowed or not.

"""


class RAG:
    def __init__(self, corpus, client):
        self.client = client
        self.search_index = SearchIndex(corpus, chunk_fn=no_chunking)
        self.search_index.build_index()
        self.reset()
        
    def guardrail(self, message: str):
        inputs = [
            {"role": "system", "content": guardrail_system_prompt},
            {"role": "user", "content": message}
        ]
        resp = openai.responses.create(
            model="gpt-3.5-turbo",
            input=inputs,
        )
        if "BLOCKED" in resp.output_text:
            return True
        return False

    def reset(self):
        self.context = [{'role': 'system', 'content': answer_system_prompt}]

    def query_rewrite(self, message):
        response = self.client.responses.create(
            model='gpt-3.5-turbo',
            input=[
                {'role': 'system', 'content': query_system_prompt},
                {'role': 'user', 'content': message},
            ],
        )
        return response.output_text.strip()

    def chat(self, message):
        if self.guardrail(message):
            return "BLOCKED: This question does not pertain to search or information retrieval."

        query = self.query_rewrite(message)
        results = self.search_index.search(query)
        snippets = '\n\n---\n\n'.join(snippet for _, snippet in results)
        context = deepcopy(self.context)
        context.append({'role': 'user', 'content': message})
        context.append({
            'role': 'user',
            'content': "Use these retrieved snippets from Doug's blog:\n\n" + snippets,
        })
        response = self.client.responses.create(
            model='gpt-3.5-turbo',
            input=context,
        )
        answer = response.output_text
        self.context.append({'role': 'user', 'content': message})
        self.context.append({'role': 'assistant', 'content': answer})
        return answer, query, results

In [ ]:
prebaked_messages = [
    'What does Doug say is the best way to use BM25?',
    "Explain vector search in Doug's voice.",
]

rag = RAG(corpus, openai)
for question in prebaked_messages:
    answer, query, results = rag.chat(question)
    display(Markdown(
        f'**Question:** {question}\n\n**Rewritten query:** {query}\n\n**Answer:** {answer}'
    ))
    display(Markdown('### Evidence supplied to the model'))
    for title, snippet in results:
        display(Markdown(f'**{title}**\n\n{snippet}'))
    display(HTML("</hr>"))

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

**Question:** What does Doug say is the best way to use BM25?

**Rewritten query:** Best practices for utilizing the BM25 algorithm

**Answer:** Doug believes that the best way to use BM25 is to leverage the BM25F approach, which is a multi-field BM25 calculation. By combining the BM25 scores across different fields, you can account for scenarios where a term may be common in one field but rare in another. This blending of document frequencies enables a more accurate representation of term specificity across multiple fields. Utilizing Lucene's BlendedTermQuery can assist in achieving this multi-field BM25 calculation. Doug suggests implementing BM25F by considering the document frequency blending aspect and scaling term frequencies to account for document length variations in different fields.

Doug also acknowledges the challenges with naive BM25 scoring in fielded searches, where term frequencies across different fields may lead to inaccurate results. By normalizing term frequencies based on field length and applying saturation curves, BM25F incorporates a more refined approach to multi-field scoring. Combining the blended IDF with properly normalized term frequencies results in a more accurate BM25F score calculation across multiple fields. 

Ultimately, Doug emphasizes the importance of understanding the intricacies of BM25F and its components to ensure a more precise and meaningful search relevance calculation.

### Evidence supplied to the model

**BM25F in Lucene with BlendedTermQuery**

 As part of the London hack days Diego Ceccarelli started a BM25F implementation. I began to continue it at Lucene Revolution’s Lucene hackathon. I realized though that when you break down the problem, BM25F can be implemented using existing Lucene bits, including the existing BM25Similarity and the BlendedTermQuery. 

 BM25 and BM25F 

 I’ve written about BM25 before. BM25 is an iteration on the classic TF*IDF ranking formula, but with several innovations: 

 
   Term Frequency Saturation: We know more times a search term occurs in a document, the more we should consider this document relevant for that search term. But with BM25, we realize at some point you reach diminishing returns. BM25 reaches this point relatively quickly, compared to classic TF*IDF. 
   Smarter document length weighting: A search term occurring once in a short doc is more relevant than a single term occurring in a longer doc (a book). BM25 penalizes/rewards document length relative to a document’s average document length, as opposed to just having a constant multiple based on document length 
   Basically the same IDF calculation: Rare search terms (low doc freq; high IDF) get weighted more heavily than common search terms. The BM25 calculation isn’t that much different than TF*IDF’s calculation 
 

 BM25F performs per-field BM25 calculation, but uses the shared document frequency across multiple fields. This document frequency blending is important. For example, sometimes you have scenarios where a common term, say “cat” is actually rare in one particular field (say the “title” field). But when you broaden out to other fields, you then realize that cat is particularly common, and shouldn’t be scored so highly. 

 In other words, BM25F basically does 

 CombinedIDF * ( BM25_title + BM25_description + …)
 

 (note there are all kinds of subtle variants) 

 Can we implement that using existing Lucene bits? I think so… with a few caveats that Diego has pointed out. You can follow along by looking at this github repo with the sample code from this post. 

 Per-field BM25Similarity 

 BM25F lets us configure BM25 parameters per field. Luckily, per-field similarity is pretty easy to configure in Lucene using a PerFieldSimilarityWrapper. We simply need to setup our index accordingly. Notice in the similarity below, k1 and b differ for title and description: 

 java
static Similarity perFieldSimilarities =  new PerFieldSimilarityWrapper() {
      @Override
      public Similarity get(String name) {
          if (name.equals("title")) {
              return new BM25FSimilarity(/*k1*/1.2f, /*b*/0.8f);
          } else if (name.equals("description")) {
              return new BM25FSimilarity(/*k1*/1.4f, /*b*/0.9f);
          }
          return new BM25FSimilarity();
      }
};
 

 Then when we setup the IndexReader 

 ```java
IndexWriterConfig config = new IndexWriterConfig(analyzer);
config.setSimilarity(perFieldSimilarities); 

 IndexWriter w = new IndexWriter(index, config);
```` 

 BlendedTermQuery to implement BM25F 

 Lucene’s BlendedTermQuery forms the guts behind Elasticsearch’s cross_field search. What BlendedTermQuery does is blend global term stats as best as possible across different fields. If the document frequency for cat in title is 3, but the document frequency for cat in the description field is 50, it takes the maximum, 50. This approximates the actual document frequency across both fields (we can’t figure out overlaps very efficiently at query time). 

 This value is then used for searching both title and description, accounting for the true rareness of the term. To do this, it builds a set of Lucene TermQueries and tells the TermQuery for each field to use the blended document frequency, instead of what’s in the index for that field. So you can imagine that now you have two queries description:cat (\*with doc freq 50) and title:cat (\*with doc freq 50) 

 BlendedTermQuery then gives you two options for combining these queries. You can either as a dismax query (take the max of the underlying field scores) or a boolean query (sum the underlying field scores). In other words, you can pick between one of two ranking functions: 

 
   Dismax: The best scoring field wins, with an optional tie-breaker parameter to incorporate other field scores. In other words: max( description:cat (with doc freq 50), title:cat (with doc freq 50), …) 
   Boolean: A summation of the field scores, in other words description:cat (with doc freq 50) + tilte:cat (with doc freq 50) + … 
 

 Examining the math above, we really care about the latter form for BM25F as it involves a summation. To implement that, we simply use the following code. 

 ```java
BlendedTermQuery query = new BlendedTermQuery.Builder()
				 .add(new Term(“title”, “cat”), /boost/1.0f)
				 .add(new Term(“description”, “cat”), /boost/1.0f)
				 .setRewriteMethod(BlendedTermQuery.BOOLEAN_REWRITE)
				 .build(); 

 ``` 

 Checking the ranking function 

 So with the BlendedTermQuery above, what we have now is something like the following ranking function 

 description:cat (with BM25, docfreq=50) + title:cat (with BM25, docfreq=50)
 

 Which is the same as 

 IDF( docfreq=50) * (description:cat with BM25) +  IDF(docfreq=50) * (title:cat with BM25) + 
 

 Here description:cat with BM25 means the non-IDF portion of the BM25 calculation (the term frequency and length part, specific to this field, boosts for this field, etc). The IDF(docfreq=X) is the BM25 IDF formula for a term with a given document frequency. 

 Now it turns out, this is (pretty much) BM25F! All we need to do to make this BM25F is to factor the IDF (docfreq=50) out to see the formula we have above 

 IDF( docfreq=50) * ( (description:cat with BM25) +  (title:cat with BM25) )
 

 A few subtle differences 

 There’s a few reasons this isn’t quite BM25F. First of all Lucene’s boolean query uses a coordinating factor (coord) to reward/punish documents that match all the clauses. So: 

 IDF( docfreq=50) * ( (description:cat with BM25) +  (title:cat with BM25) )
 

 Is actually 

 coord * IDF( docfreq=50) * ( (description:cat with BM25) +  (title:cat with BM25) )
 

 If you recall coord is the number of matched clauses / number of total clauses. So if only 1 out of 2 clauses match, coord is ½. This further rewards documents that match more than one clause. Latest versions of Lucene have removed coord 

 Another thing to point out is that it’s difficult to piece together the true cross-field document frequency at query time. So the maximum is taken as an approximation. The actual blended document frequency isn’t entirely accurate, but we know it ranges somewhere between the max of the two field doc frequencies (when there’s 100% overlap) and the sum (when the fields mention the term in different documents). 

 And that’s it! Get in touch. 

 That’s it! Get in touch – I’d love to hear feedback if I missed anything. And if you want to pick my brain for free check out the lunch and learns. I’m about to announce some new topics that might interest your team! And as always, don’t hesitate to reach out about our relevancy services – we just got this great testimonial from Careerbuilder about our services: 

 
   OSC’s Solr/Lucene knowledge expanded and greatly improved the abilities of our public job search. They delivered technical excellence at every turn: demonstrating expertise in Lucene internals, relevance models, and data science backed by a solid methodology for improving search relevance. On a deliverables front: they learned our legacy search stack quickly and made high-quality code contributions. OSC marries technical excellence with strategic insight: we highly recommend their experts to any search team. 
 


**BM25F from scratch**

Expanding on my [Cheat at Search Essentials](https://softwaredoug.com/blog/2025/07/31/cheat-at-search-essentials) training class, I decided to go beyond just plain-old-BM25, to multi-field BM25, or [BM25F](https://ir.webis.de/anthology/2004.cikm_conference-2004.6).

Search engines like Elasticsearch use BM25 to rank search results. BM25 answers an important question — IF we see a query term match, how much is that passage about the matched term? That’s not all of what search relevance is, but it’s an important part. This question underlies not just BM25, but emerging sparse models (which themselves might need to retrieve across fields!).

## Get to know the BM25 building blocks

[Numerous](https://www.elastic.co/blog/found-bm-vs-lucene-default-similarity) [articles](https://opensourceconnections.com/blog/2015/10/16/bm25-the-next-generation-of-lucene-relevation/) go into great depth on BM25 and why its the golden baseline for search. But I’ll work to capture the building blocks a bit differently. We’ll take these pieces and use them to build BM25 for multiple fields. 

First realize most search systems, for years, scored results using three primary inputs:

| Factor | Definition | Impact to score |
| --- | --- | --- |
| Term Frequency | a word count, how often does `skywalker` occur in this document? | Higher (more frequent) is good! |
| Document Frequency | rareness, if searching for `luke` OR `skywalker` , `luke` is common/ambiguous while `skywalker` is rare/specific.  | Lower (more specific) is good! |
| Field Length | number of terms in a field (matches or otherwise).  | Matches on shorter fields seen as more important than longer |

We get the name `TF*IDF` from `TF / DF` from these stats. 

Throughout history, we have learned how to best use these ingredient; lessons now baked into BM25. Lessons that came from countless studies, across numerous domains, of how users view the relevance of term matches.

Getting these basics down, we’ll then see how we mix them together to go beyond a single field, to build up BM25F with multiple fields.

### BM25’s Document frequency

First, we know a bit how users perceive a [terms *specificity*](https://www.staff.city.ac.uk/~sbrp622/idfpapers/ksj_orig.pdf). We usually say a term that rare in the corpus is more specific to the users intent. A rare terms occurrence in an irrelevant document  is less probable than a common one. 

We’ve learned users notion of specificity isn’t linear. A term occurring in 2x fewer docs is not 2x closer to the users intent. 

Notice below, in BM25’s IDF scoring, for every 100 raw document frequency, we don’t get a linear decrease in specificity. At first the drop off is steep. Then it tapers.

![image.png](/assets/media/bm25f-from-scratch/image.png)

BM25 decays specificity logarithmically, IE this formula seems to work best:

```python
def compute_idf(num_docs, df):
    """Calculate idf score from num_docs (index size) and df (a term's doc freq)""" 
    return np.log(1 + (num_docs - df + 0.5) / (df + 0.5))
```

When we get to multiple fields, we’ll need to account for every fields differing document frequency stats. More on that below. 

### Scaling term frequency to document length

Pickup a copy of Proust’s [In Search of Lost Time](https://www.britannica.com/topic/Massive-Tomes-10-of-the-Worlds-Longest-Novels) and you’ll have to read maybe a million words? Read a tweet, you’ll encounter maybe a dozen?

If Proust happened to use the term `skywalker` it’d be pretty weird. But maybe not entirely unexpected to happen accidentally once in 4000 pages. A term frequency of “1” in Proust should be seen as low-information relative to the term occurring a thousand times. We would not say the book is “about” `skywalker` or Star Wars by one mention.

If a tweet mentions `skywalker` once, wow, that one occurrence is very important to the tweet! It’s like 10% of the information! The tweet is very much about `skywalker` 

BM25 scales term frequency so that long document’s TF counts less than short documents:

![image.png](/assets/media/bm25f-from-scratch/image 1.png)

It’s a simple formula, with parameter `b` that changes how much document length influences the term freq score:

```python
def scaled_tf(term_freq, doc_len, avg_doc_len, b=0.8):
	(term_freq) / (1 - b + b * doc_len / avg_doc_len)
```

With multiple fields, each field has a different length. We’ll have to account for each filelds length independently. 

## Term frequency saturation

As a passage mentions a term more-and-more times, users don’t think its relevance goes higher and higher, it plateaus. 

You’re reading a blog post, you see the word `bm25` mentioned. Oh wow maybe this article is about `bm25`? You see it a second time - confirming your suspicions. Then a third, fourth, fifth, … 99th time. By the 99th time you see `bm25` in the article, it can’t get much more relevant to the term. We get it by now.

Information Retrieval experiments confirm, relevance perceptions reach of point of saturation after they've seen a term enough. In other words, like document frequency, term frequency starts off strong with each incremental mention of a term, but gradually peters out.

![image.png](/assets/media/bm25f-from-scratch/image 2.png)

The math for this is simple, with a constant k1 controlling the rate of saturation:

```python
def bm25_tf(termfreq, k1=1.2):
	return termfreq / (termfreq + k1)
```

(Note I’m using the Lucene formulation which removes the k1 + 1 in the numerator for performance as it doesn’t impact ordering)

### BM25 put together

Now we need to compute `TF * IDF` but BM25 style, by putting together everything from below. 

IDF we can just take from above.

But the “TF” part we combine the `scaled_tf` and the `bm25_tf` , giving the classic BM25 TF formula:

```python
def bm25_scaled_tf(term_freq, doc_len, avg_doc_len, b=0.8):
	(term_freq) / (termfreq + k1 * (1 - b + b * doc_len / avg_doc_len))
```

The math here saturates with a tf / (tf + constant). That constant is tuned to the document length (it’s the bit that bends the line above). But instead of bending a line, we’re bending the saturation curve. 

Multifield TF*IDF requires us to derive a kind of TF and a kind of IDF for a term across multiple fields. First we will start with IDF. 

## BM25 Problems with fielded search - specificity

Almost all full-text search engines break the index into mini-indices we search called ***fields***. Like for books maybe `title`, `body`, or `description`. Each field lives in its own universe of scoring statistics (average document length, term and document frequencies, etc).

It turns out, raw BM25 has a few problems when text is split across fields.

In a technical book search I worked on, some book publishers would put “book” in the title. It wasn’t common. But try to imagine what BM25 would do when a user searched for `javascript book` . Thousands of technical books have `javascript` in the title. A few, spurious books have `book` in the title. 

Because `book`s high IDF, you’d get weird results with no Javascript! Such as:

Query: Javascript Book

1. The big **book** on squirrels 
2. C Programmers **Book** about Pointers
3. The missing iPhone **book**

The root of the problem: document frequency of `book` in the `title` field doesn’t reflect its actual specificity in the full corpus. If we looked at other fields like `body` and `description` we might (though still not fullproof!) get a better sense of `book`'s specificity.

This is the first spot where BM25F comes to help. If we were to naively sum BM25 scores across fields, we would end up with the busted results above, strongly biased towards the rare title matches.

```python
score = TF('title', 'book') * IDF('title', 'book') \
        + TF('body', 'book') * IDF('body', 'book')
```

A better way to model specificity, would be to somehow know the true document frequency across fields, then compute IDF score using that. Usually, this is done by simply taking the max document frequency.

```python
combined_doc_freq = max(DF('title', 'book'), DF('body', 'book'))
blended_idf = compute_idf( corpus_len, combined_doc_freq)
```

Elasticsearch has a query mode called [`cross_fields`](https://www.elastic.co/docs/reference/query-languages/query-dsl/query-dsl-multi-match-query#type-cross-fields) . It does just this combining part. Elasticsearch cross_fields blends document frequencies, then repeats the math from above:

```python
score = TF('title', 'book') * blended_idf \
        + TF('body', 'book') * blended_idf
```

It’s a good-enough 80% solution that captures the biggest problem with BM25 across fields. But it’s not all the way to BM25F.

## Multi field TF double counts

Another problem with naive BM25 summing occurs with the combination of term frequencies. If we just blend document frequencies, but leave term frequencies alone, we let each term saturate independently.

```python
score = TF('title', 'book') * blended_idf \
        + TF('body', 'book') * blended_idf 
```

Here we might see `title:book` match once, and count it very strongly (it’s early in the saturation curve). While `body:book` is mentioned hundreds of times, but we also count its early trips in the saturation curve.

| Match of body | Impact | Notes |
| --- | --- | --- |
| Title (tf=1) | High | Early in saturation curve |
| Body (tf=100) | High but diminishing | Farther in saturation curve |
| **Total** | 2 x High + diminishing | double counted |

In reality what we want to see is something like

| Match of body | Impact | Notes |
| --- | --- | --- |
| Title + Body (tf=101) | High but diminishing |  |
| **Total** | High but diminishing |  |

BUT and this is a big but 🍑

Comparing these term frequencies is like an apples to oranges comparison. Remember how document length modulates term frequency. The 1 term freq in short title should count much more than the 100 terms in the body. Maybe we’d say that title term freq should be scaled up. While the body scaled down.

So looking at a length-normalized situation, what we’d *actually* want is maybe the following (with fake, hand waving scaling up/down)

| Match of body | Impact | Notes |
| --- | --- | --- |
| Title (tf 1 → 5) + Body (tf=101 → 30) | High but diminishing |  |
| **Total:**  35 | High but diminishing |  |

So that’s where we apply these steps

1. The length normalization step to scale up / down each field’s term freq → `scaled_tf` from above
2. Apply saturation to the output of this (`bm25_tf`)

The final combined “TF” becomes:

```python
bm25_tf(scaled_tf(TF('title', 'book'), title_len, avg_title_len) + 
        scaled_tf(TF('body', 'book'), body_len, avg_body_len))
```

So the final BM25F looks like:

```python
combined_doc_freq = max(DF('title', 'book'), DF('body', 'book'))
blended_idf = compute_idf( corpus_len, combined_doc_freq)

bm25f_tf = bm25_tf(scaled_tf(TF('title', 'book'), title_len, avg_title_len) + 
                   scaled_tf(TF('body', 'book'), body_len, avg_body_len))
                   
                   
bm25f = blended_idf * bm25f_tf
```

And that’s it, now you know BM25F, and maybe even have a deeper intuition about BM25 itself!

What questions do you have about sparse document scoring? How would we do a multi-field search for a sparse transformer model? Is document frequency the only / best way to compute term specificity? How do we extract our tokens from the raw strings? Tokenization is another important design point in a lexical system to accurately reflect when a 'match' truly occurs.

A lot of open questions still and room for you to innovate and tune in lexical. Get in touch if you'd like to chat more!

**Ugly hack to force BM25 to 0-1**

Its convenient to have a lexical score normalized from 0-1. Sadly BM25 scores tend to be all over the place (0.5? 5.1? 12.51?). Fine for ranking. Annoying for other goals. That's why I wrote a post about [one way to compute probabilities from BM25](https://softwaredoug.com/blog/2026/03/06/probabilistic-bm25-utopia).

In that post, I allude to one hack that forces BM25 to 0-1. Let's walk through it.

A query term’s BM25 score is IDF \* TF.

**Lucene’s TF is already normalized**

Lucene drops the (k1 + 1) in the numerator of BM25, giving you:

![](/assets/media/daily-search-tips/23195748/cb18e3c42d42e1a3f582175c60904b9f.png)

​

![]()

​

Now we’ve got a TF term bounded from 0-1. Nice.

**Now we’ve got to tackle IDF.** Here’s the standard IDF. Here N: num docs in corpus; n: doc frequency of this term.

![](/assets/media/daily-search-tips/23195748/1ddb855665ecf45de4555e2025f0cfc6.png)

​

Turns out the [max value of this function is log(N)](https://www.desmos.com/calculator/keh0iej0zl). So! just slap a log(N) denominator under that sucker. Now that too has become 0-1.

BM25 now ranges 0-1, while preserving in-query ranking.

**It’s 100% a hack.** What’s nice or problematic about this:

* 👎 You’re multiplying small numbers. Most IDFs/TF terms will be low, making the final result low
* 👍 You don’t need to know anything about distribution of BM25 scores (ie [to do min/max or z-score normalization](https://www.codecademy.com/article/min-max-zscore-normalization))
* 🤙 No calibration to try to be a proper probability. 0.5 BM25 here doesn’t equate to midpoint of your relevance labels, [like in BB25.](https://github.com/cognica-io/bayesian-bm25)​
* 🆗 Combining with other terms means dividing by number of query terms to stay between 0-1

Use with care.

-Doug

**AI Powered Search training - late signup available** - <http://maven.com/search-school/ai-powered-search>​

​

*This is part of Doug's Daily Search tips - [subscribe here](http://softwaredoug.kit.com)*

Compare the rewritten query and retrieved results with `Broken-RAG.ipynb`. This isolates query rewriting as one change; guardrails, chunking changes, and other retrieval strategies are intentionally left for later experiments.